In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

In [3]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = openrouter_api_key

# Consider using an enum for this.
MODEL_MAP = {
    "CLAUDE": {
        "model": "claude-3-5-sonnet-20240620",
        "api_key": anthropic_api_key,
        "endpoint": "https://api.anthropic.com/v1"
    },
    "GPT": {
        "model": "gpt-4o-mini",
        "api_key": OPENROUTER_API_KEY,
        "endpoint": OPENROUTER_BASE_URL,
    },
    "Grok": {
        "model": "grok-beta",
        "api_key": grok_api_key,
        "endpoint": "https://api.grok.com/v1"
    },   
    "DeepSeek":{
        "model": "deepseek-reasoner",
        "api_key": ds_api_key,
        "endpoint": "https://api.deepseek.com/v1",
    },
    "Google": {
        "model": "gemini-2.0-flash-exp",
        "api_key": google_api_key,
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai"
    },
}

In [4]:
def call_openai(model, messages, temperature=0.7):
    client = OpenAI(
        base_url=MODEL_MAP[model]["endpoint"],
        api_key=MODEL_MAP[model]["api_key"]
    )
    response = client.chat.completions.create(
        model=MODEL_MAP[model]["model"],
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


In [5]:
SYSTEM_PROMPT = """
You are an expert medical data scientist generating high-fidelity, synthetic patient health records for research purposes.
Your goal is to create data that mimics realistic, complex health patterns, including correlations between lifestyle, demographics, and clinical lab values.
The output must be in JSON or comma-separated CSV format.
Ensure that all data points are plausible (e.g., age 80 does not have a resting heart rate of 30, BMI of 40 correlates with high blood pressure, etc.).
Do not include any real patient identifiers (PHI).
Generate 20 distinct, uncorrelated features per record.
"""

USER_PROMPT = """
Role: Act as an expert data scientist and clinical informaticist specializing in generating high-fidelity, privacy-compliant synthetic healthcare data.
Task: Generate a CSV-formatted dataset of 100 synthetic patient records. 
Dataset Specifications:

    Target: Individuals' health conditions and associated metrics.
    Format: CSV, comma-separated.
    Privacy: No real PII (Personally Identifiable Information). Use realistic but artificial data.
    Data Types: Include a mix of numerical, categorical, and binary variables.
    Realism: Ensure logical correlations between features (e.g., higher BMI correlates with higher blood pressure, older age correlates with higher prevalence of chronic conditions). 

Features to Include (20 total):

    Patient_ID (Unique alphanumeric)
    Age (Integer, 18-90)
    Gender (Categorical: Male, Female, Other)
    Height_cm (Numerical)
    Weight_kg (Numerical)
    BMI (Calculated)
    BloodPressure_Systolic (Numerical)
    BloodPressure_Diastolic (Numerical)
    HeartRate_BPM (Numerical)
    Smoking_Status (Categorical: Never, Former, Current)
    Alcohol_Consumption (Categorical: Low, Medium, High)
    Physical_Activity_Hours_Per_Week (Numerical)
    HbA1c_Level (Numerical, 4.0 - 10.0)
    LDL_Cholesterol_mgdL (Numerical)
    Daily_Sleep_Hours (Numerical)
    Known_Allergies (Categorical: None, Penicillin, Peanuts, Pollen)
    Primary_Condition (Categorical: Hypertension, Type 2 Diabetes, Asthma, None)
    Number_of_Medications (Integer)
    Mental_Health_Score_PHQ9 (Integer, 0-27)
    Hospital_Readmission_30d (Binary: 0 or 1) 

Output Format:
Provide the output directly in CSV format.
"""

messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ]

In [6]:
def generate_dataset(model):
    return call_openai(model, messages)

In [7]:
csv_string = generate_dataset("GPT")
csv_string = csv_string.replace("```", "").replace("csv", "").strip()
print(csv_string)

Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,BloodPressure_Systolic,BloodPressure_Diastolic,HeartRate_BPM,Smoking_Status,Alcohol_Consumption,Physical_Activity_Hours_Per_Week,HbA1c_Level,LDL_Cholesterol_mgdL,Daily_Sleep_Hours,Known_Allergies,Primary_Condition,Number_of_Medications,Mental_Health_Score_PHQ9,Hospital_Readmission_30d
P001,34,Male,175,70,22.86,120,80,72,Never,Low,5,5.8,110,7,None,None,2,4,0
P002,58,Female,160,80,31.25,140,90,75,Former,Medium,3,6.5,150,6,Peanuts,Hypertension,3,10,1
P003,45,Other,170,85,29.41,130,85,68,Current,High,2,7.2,160,6,Pollen,Type 2 Diabetes,4,12,0
P004,68,Male,180,90,27.78,150,95,70,Never,Medium,3,6.2,140,8,None,Hypertension,1,8,1
P005,22,Female,165,55,20.20,115,75,80,Never,Low,10,5.4,90,8,Pollen,None,1,3,0
P006,50,Male,175,95,31.02,145,92,76,Former,Medium,1,7.0,170,5,None,Type 2 Diabetes,5,15,1
P007,39,Female,170,65,22.49,125,80,74,Never,Low,6,5.6,100,7,Penicillin,None,2,2,0
P008,82,Male,160,85,33.29,160,100,60,Current,High,1,8.0,180,5,None,Hyperten

In [8]:
import pandas as pd
import io

# Sample data
csv_data_stream = io.StringIO(csv_string)
# df = pd.DataFrame(csv_data)

csv_data = None

def load_csv_data():
    global csv_data

    csv_data = pd.read_csv(csv_data_stream)

load_csv_data()

In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import datetime
import warnings
import tempfile
from cachetools import cached, TTLCache

warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")


cache = TTLCache(maxsize=128, ttl=300)

@cached(cache)
def get_gender_categories():
    global csv_data
    if csv_data is None:
        return []
    cats = sorted(csv_data['Gender'].dropna().unique().tolist())
    cats = [cat.capitalize() for cat in cats]
    return cats

def get_age_range():
    global csv_data
    if csv_data is None or csv_data.empty:
        return None, None
    return csv_data['Age'].min(), csv_data['Age'].max()


def filter_data(start_age, end_age, category):
    global csv_data

    if isinstance(start_age, str):
        start_age = start_age.strip()
    if isinstance(end_age, str):
        end_age = end_age.strip()

    df = csv_data.loc[(csv_data['Age'] >= start_age) & (csv_data['Age'] <= end_age)].copy()

    if category != "All Categories":
        df = df.loc[df['Gender'].str.capitalize() == category].copy()

    return df

In [10]:
def get_dashboard_stats(start_age, end_age, category):
    df = filter_data(start_age, end_age, category)
    if df.empty:
        return (0, 0, 0, "N/A")

    df['Body_Mass_Index'] = df['Weight_kg'] / (df['Height_cm'] / 100) ** 2
    total_weight = df['Weight_kg'].sum()
    total_height = df['Height_cm'].sum()
    total_patients = df['Patient_ID'].nunique()
    avg_weight = total_weight / total_patients if total_patients else 0
    avg_height = total_height / total_patients if total_patients else 0

    gender_weight = df.groupby('Gender')['Weight_kg'].sum().sort_values(ascending=False)
    top_gender = gender_weight.index[0] if not gender_weight.empty else "N/A"

    return (total_weight, total_patients, avg_weight, top_gender.capitalize())

In [11]:
def get_data_for_table(start_age, end_age, category):
    df = filter_data(start_age, end_age, category)
    if df.empty:
        return pd.DataFrame()

    df = df.sort_values(by=["Patient_ID", "Age"], ascending=[True, False]).copy()

    columns_order = [
        "Patient_ID", "Age", "Gender", "Height_cm", "Weight_kg", "BMI", "BloodPressure_Systolic", 
        "BloodPressure_Diastolic", "HeartRate_BPM", "Smoking_Status", "Alcohol_Consumption", 
        "Physical_Activity_Hours_Per_Week", "HbA1c_Level", "LDL_Cholesterol_mgdL", "Daily_Sleep_Hours", 
        "Known_Allergies", "Primary_Condition", "Number_of_Medications", "Mental_Health_Score_PHQ9", 
        "Hospital_Readmission_30d"
    ]
    columns_order = [col for col in columns_order if col in df.columns]
    df = df[columns_order].copy()

    df['BMI'] = df['Weight_kg'] / (df['Height_cm'] / 100) ** 2
    return df

In [12]:
def get_plot_data(start_age, end_age, category):
    df = filter_data(start_age, end_age, category)
    if df.empty:
        return pd.DataFrame()
    df['BMI'] = df['Weight_kg'] / (df['Height_cm'] / 100) ** 2
    plot_data = df.groupby(df['Age'].dt.date)['BMI'].mean().reset_index()
    plot_data.rename(columns={'Age': 'date'}, inplace=True)
    return plot_data

In [13]:
def get_bmi_by_gender(start_age, end_age, category):
    df = filter_data(start_age, end_age, category)
    if df.empty:
        return pd.DataFrame()
    df['BMI'] = df['Weight_kg'] / (df['Height_cm'] / 100) ** 2
    cat_data = df.groupby('Gender')['BMI'].mean().reset_index()
    cat_data = cat_data.sort_values(by='BMI', ascending=False)
    return cat_data

In [14]:
def get_high_bmi_patients(start_age  , end_age, category):
    df = filter_data(start_age, end_age, category)
    if df.empty:
        return pd.DataFrame()
    df['BMI'] = df['Weight_kg'] / (df['Height_cm'] / 100) ** 2
    prod_data = df.groupby('Gender')['BMI'].mean().reset_index()
    prod_data = prod_data.sort_values(by='BMI', ascending=False).head(10)
    return prod_data

In [15]:
def create_matplotlib_figure(data, x_col, y_col, title, xlabel, ylabel, orientation='v'):
    plt.figure(figsize=(10, 6))
    if data.empty:
        plt.text(0.5, 0.5, 'No data available', ha='center', va='center')
    else:
        if orientation == 'v':
            plt.bar(data[x_col], data[y_col])
            plt.xticks(rotation=45, ha='right')
        else:
            plt.barh(data[x_col], data[y_col])
            plt.gca().invert_yaxis() 

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()

    with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as tmpfile:
        plt.savefig(tmpfile.name)
    plt.close()
    return tmpfile.name

In [16]:
def update_dashboard(start_age, end_age, category):
    total_weight, total_patients, avg_weight, top_gender = get_dashboard_stats(start_age, end_age, category)

    # Generate plots
    bmi_data = get_plot_data(start_age, end_age, category)
    bmi_by_gender_data = get_bmi_by_gender(start_age, end_age, category)
    high_bmi_patients_data = get_high_bmi_patients(start_age, end_age, category)

    bmi_over_age_path = create_matplotlib_figure(
        bmi_data, 'age', 'BMI',
        "BMI Over Age", "Age", "BMI"
    )
    bmi_by_gender_path = create_matplotlib_figure(
        bmi_by_gender_data, 'Gender', 'BMI',
        "BMI by Gender", "Gender", "BMI"
    )
    high_bmi_patients_path = create_matplotlib_figure(
        high_bmi_patients_data, 'Gender', 'BMI',
        "High BMI Patients", "BMI", "Gender", orientation='h'
    )

    # Data table
    table_data = get_data_for_table(start_age, end_age, category)

    return (
        bmi_over_age_path,
        bmi_by_gender_path,
        high_bmi_patients_path,
        table_data,
        total_weight,
        total_patients,
        avg_weight,
        top_gender
    )

In [21]:
def create_dashboard():
    min_age, max_age = get_age_range()
    if min_age is None or max_age is None:
        min_age = 18
        max_age = 100

    default_start_age = min_age
    default_end_age = max_age

    with gr.Blocks(css="""
        footer {display: none !important;}
        .tabs {border: none !important;}  
        .gr-plot {border: none !important; box-shadow: none !important;}
    """) as dashboard:
        
        gr.Markdown("# BMI Dashboard")

        # Filters row
        with gr.Row():
            start_age = gr.Number(
                label="Start Age",
                value=default_start_age
            )
            end_age = gr.Number(
                label="End Age",
                value=default_end_age
            )
            category_filter = gr.Dropdown(
                choices=["All Categories"] + get_gender_categories(),
                label="Category",
                value="All Categories"
            )

        gr.Markdown("# Key Metrics")

        # Stats row
        with gr.Row():
            total_weight = gr.Number(label="Total Weight", value=0)
            total_patients = gr.Number(label="Total Patients", value=0)
            avg_weight = gr.Number(label="Average Weight", value=0)
            top_gender = gr.Textbox(label="Top Gender", value="N/A")

        gr.Markdown("# Visualisations")
        # Tabs for Plots
        with gr.Tabs():
            with gr.Tab("BMI Over Age"):
                bmi_over_age_image = gr.Image(label="BMI Over Age", container=False)
            with gr.Tab("BMI by Gender"):
                bmi_by_gender_image = gr.Image(label="BMI by Gender", container=False)
            with gr.Tab("High BMI Patients"):
                high_bmi_patients_image = gr.Image(label="High BMI Patients", container=False)

        gr.Markdown("# Raw Data")
        # Data Table (below the plots)
        data_table = gr.DataFrame(
            label="BMI Data",
            type="pandas",
            interactive=False
        )

        # When filters change, update everything
        for f in [start_age, end_age, category_filter]:
            f.change(
                fn=lambda s, e, c: update_dashboard(s, e, c),
                inputs=[start_age, end_age, category_filter],
                outputs=[
                    bmi_over_age_image, 
                    bmi_by_gender_image, 
                    high_bmi_patients_image,
                    data_table,
                    total_weight, 
                    total_patients,
                    avg_weight, 
                    top_gender
                ]
            )

        # Initial load
        dashboard.load(
            fn=lambda: update_dashboard(default_start_age, default_end_age, "All Categories"),
            outputs=[
                bmi_over_age_image, 
                bmi_by_gender_image, 
                high_bmi_patients_image,
                data_table,
                total_weight, 
                total_patients,
                avg_weight, 
                top_gender
            ]
        )

    return dashboard

In [ ]:
dashboard = create_dashboard()
dashboard.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/thomas/Desktop/llm_engineeri